<a href="https://colab.research.google.com/github/VainaviS/EndoNet/blob/main/notebooks/MaskRcNNv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
!pip install torch torchvision torchaudio
!pip install opencv-python matplotlib
!pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

zip_name = list(uploaded.keys())[0]

extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [ ]:
import json
import random
from sklearn.model_selection import train_test_split

coco_path = "/content/dataset/Glenda_v1.5_classes/coco.json"

with open(coco_path) as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]

# split images
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

train_ids = set([img["id"] for img in train_imgs])
val_ids = set([img["id"] for img in val_imgs])

# split annotations
train_anns = [ann for ann in annotations if ann["image_id"] in train_ids]
val_anns   = [ann for ann in annotations if ann["image_id"] in val_ids]

In [ ]:
# after splitting
def oversample(images, annotations, target_class_id=4, factor=4):
    extra_imgs = []
    extra_anns = []

    img_map = {img["id"]: img for img in images}

    for ann in annotations:
        if ann["category_id"] == target_class_id:
            img = img_map[ann["image_id"]]
            for _ in range(factor):
                extra_imgs.append(img)
                extra_anns.append(ann)

    return images + extra_imgs, annotations + extra_anns

In [ ]:
def save_coco(images, annotations, path):
    coco_out = {
        "info": coco["info"],
        "licenses": coco.get("licenses", []),
        "images": images,
        "annotations": annotations,
        "categories": coco["categories"]
    }
    with open(path, "w") as f:
        json.dump(coco_out, f)

os.makedirs("/content/split/train", exist_ok=True)
os.makedirs("/content/split/val", exist_ok=True)

save_coco(train_imgs, train_anns, "/content/split/train.json")
save_coco(val_imgs, val_anns, "/content/split/val.json")

In [ ]:
train_images = "/content/dataset/Glenda_v1.5_classes/frames"
val_images   = "/content/dataset/Glenda_v1.5_classes/frames"

train_json = "/content/split/train.json"
val_json   = "/content/split/val.json"

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog

# remove datasets if already registered
if "glenda_train" in DatasetCatalog.list():
    DatasetCatalog.remove("glenda_train")
    MetadataCatalog.remove("glenda_train")

if "glenda_val" in DatasetCatalog.list():
    DatasetCatalog.remove("glenda_val")
    MetadataCatalog.remove("glenda_val")

In [ ]:
from detectron2.data.datasets import register_coco_instances

register_coco_instances("glenda_train", {}, train_json, train_images)
register_coco_instances("glenda_val", {}, val_json, val_images)

In [ ]:
from detectron2.data import DatasetCatalog
from detectron2.utils.visualizer import Visualizer
import cv2, random, matplotlib.pyplot as plt

dataset_dicts = DatasetCatalog.get("glenda_train")
metadata = MetadataCatalog.get("glenda_train")

for d in random.sample(dataset_dicts, 3):
    img = cv2.imread(d["file_name"])
    v = Visualizer(img[:, :, ::-1], metadata=metadata, scale=0.5)
    out = v.draw_dataset_dict(d)
    plt.imshow(out.get_image())
    plt.show()

In [ ]:
from detectron2.config import get_cfg
from detectron2 import model_zoo

cfg = get_cfg()

cfg.merge_from_file(
    model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml")
)

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"
)

cfg.DATASETS.TRAIN = ("glenda_train",)
cfg.DATASETS.TEST = ("glenda_val",)

cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[8, 16, 32, 64, 128]]

cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00005
cfg.SOLVER.MAX_ITER = 15000
cfg.SOLVER.STEPS = []

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 5

cfg.INPUT.MIN_SIZE_TRAIN = (640, 800, 1024)
cfg.INPUT.MASK_FORMAT = "polygon"

cfg.OUTPUT_DIR = "/content/output"

In [ ]:
from detectron2.engine import DefaultTrainer
import os

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
from detectron2.evaluation import COCOEvaluator
from detectron2.data import build_detection_test_loader
from detectron2.evaluation import inference_on_dataset

evaluator = COCOEvaluator("glenda_val", cfg, False, output_dir="./output/")
val_loader = build_detection_test_loader(cfg, "glenda_val")

print(inference_on_dataset(trainer.model, val_loader, evaluator))

In [ ]:
from detectron2.engine import DefaultPredictor

cfg.MODEL.WEIGHTS = "/content/output/model_final.pth"
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # adjust if needed

predictor = DefaultPredictor(cfg)

In [ ]:
import cv2
import random
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer
from detectron2.data import DatasetCatalog, MetadataCatalog

dataset_dicts = DatasetCatalog.get("glenda_val")
metadata = MetadataCatalog.get("glenda_val")

for d in random.sample(dataset_dicts, 5):  # show 5 images
    img = cv2.imread(d["file_name"])

    outputs = predictor(img)

    v = Visualizer(img[:, :, ::-1], metadata=metadata, scale=0.8)
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    plt.figure(figsize=(8,6))
    plt.imshow(out.get_image())
    plt.title("Prediction")
    plt.axis("off")
    plt.show()

In [ ]:
from detectron2.evaluation import COCOEvaluator
from detectron2.data import build_detection_test_loader
from detectron2.evaluation import inference_on_dataset

evaluator = COCOEvaluator("glenda_val", cfg, False, output_dir="./output/")
val_loader = build_detection_test_loader(cfg, "glenda_val")

results = inference_on_dataset(trainer.model, val_loader, evaluator)

print(results)

In [ ]:
import os
import cv2
import random
import torch

from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

# class names (same order as training)
class_names = ["Peritoneum", "Ovary", "TIE", "Uterus"]

# output folder
output_root = "/content/paper_outputs"
os.makedirs(output_root, exist_ok=True)

for name in class_names:
    os.makedirs(os.path.join(output_root, name), exist_ok=True)

# load predictor
predictor = DefaultPredictor(cfg)

dataset = DatasetCatalog.get("glenda_val")
metadata = MetadataCatalog.get("glenda_val")

saved = {0:0, 1:0, 2:0, 3:0}   # count per class

for d in dataset:

    img = cv2.imread(d["file_name"])
    outputs = predictor(img)

    instances = outputs["instances"].to("cpu")

    if len(instances) == 0:
        continue

    pred_classes = instances.pred_classes.numpy()

    # check which class is present
    for c in pred_classes:

        if saved[c] < 5:   # save 5 images per class

            v = Visualizer(img[:, :, ::-1], metadata=metadata, scale=0.8)
            out = v.draw_instance_predictions(instances)

            save_path = os.path.join(
                output_root,
                class_names[c],
                f"{saved[c]}.jpg"
            )

            cv2.imwrite(save_path, out.get_image()[:, :, ::-1])

            saved[c] += 1

    # stop early if all done
    if all(v >= 5 for v in saved.values()):
        break

print("Done. Images saved in:", output_root)

In [ ]:
!zip -r paper_images.zip /content/paper_outputs

To get the final images in a grid

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.data import DatasetCatalog, MetadataCatalog

class_names = ["Peritoneum", "Ovary", "TIE", "Uterus"]

predictor = DefaultPredictor(cfg)
dataset = DatasetCatalog.get("glenda_val")
metadata = MetadataCatalog.get("glenda_val")

selected_images = {}

# pick 1 good image per class
for d in dataset:
    img = cv2.imread(d["file_name"])
    outputs = predictor(img)
    instances = outputs["instances"].to("cpu")

    if len(instances) == 0:
        continue

    pred_classes = instances.pred_classes.numpy()

    for c in pred_classes:
        if c not in selected_images:
            v = Visualizer(img[:, :, ::-1], metadata=metadata, scale=1.0)
            out = v.draw_instance_predictions(instances)
            selected_images[c] = out.get_image()

    if len(selected_images) == 4:
        break
fig, axes = plt.subplots(2, 2, figsize=(10,10))

for i, ax in enumerate(axes.flatten()):
    if i in selected_images:
        ax.imshow(selected_images[i])
        ax.set_title(class_names[i], fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.savefig("mask_rcnn_results.png", dpi=300)
plt.show()